In [1]:
import torch
from diffusion.flows.prob_paths import GaussianCondProbPath
from diffusion.training.trainer_fmm import FMMTrainer
from diffusion.sampleables.sampleable_mnist import MNISTSampleable
from diffusion.backbones.res_unet_attn import BiTimeResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(p_data=sampeable, p_simple_shape=sampeable.shape).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable, p_simple_shape=sampeable.shape
).to(device)

backbone = BiTimeResUnet(
    in_channels=sampeable.shape[0],
    channel_dims=[16, 32, 64],
    use_attention=[False, False, True],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FMMTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
    K=4,
)

In [4]:
state_dict = trainer.train(
    num_epochs=50,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=50,
    validate=True,
    plot_path="./loss.png",
)

2026-01-06 01:52:13,230 - flow-matching - INFO - Training model with size: 2.500 MiB
Epoch 0/50: 100%|██████████| 50/50 [00:14<00:00,  3.42it/s, train_loss=0.585797]
2026-01-06 01:52:28,055 - flow-matching - INFO - val loss: 0.3723926842212677, best val loss: 0.3723926842212677
Epoch 1/50: 100%|██████████| 50/50 [00:14<00:00,  3.55it/s, train_loss=0.327497]
2026-01-06 01:52:42,442 - flow-matching - INFO - val loss: 0.33942949771881104, best val loss: 0.33942949771881104
Epoch 2/50: 100%|██████████| 50/50 [00:14<00:00,  3.53it/s, train_loss=0.294730]
2026-01-06 01:52:56,863 - flow-matching - INFO - val loss: 0.31464529037475586, best val loss: 0.31464529037475586
Epoch 3/50: 100%|██████████| 50/50 [00:14<00:00,  3.51it/s, train_loss=0.265209]
2026-01-06 01:53:11,351 - flow-matching - INFO - val loss: 0.29788312315940857, best val loss: 0.29788312315940857
Epoch 4/50: 100%|██████████| 50/50 [00:14<00:00,  3.51it/s, train_loss=0.258707]
2026-01-06 01:53:25,848 - flow-matching - INFO - val

In [6]:
# torch.save(backbone.state_dict(), "./models/backbone_fmm.pt")
torch.save(state_dict, "./models/backbone_fmm_4.pt")